# Detectron2 ADE20K Interferemce Beginner's Tutorial

<img src="https://dl.fbaipublicfiles.com/detectron2/Detectron2-Logo-Horz.png" width="500">

Welcome to detectron2! This is the official colab tutorial of detectron2. Here, we will go through some basics usage of detectron2, including the following:
* Run inference on images or videos, with an existing detectron2 model
* Train a detectron2 model on a new dataset

You can make a copy of this tutorial by "File -> Open in playground mode" and make changes there. __DO NOT__ request access to this tutorial.


# Install detectron2

## 0. 사전 준비 (필수)
설치 전에 반드시 다음이 준비되어 있어야 합니다.
1.  **Visual Studio Build Tools** 설치 (Desktop development with C++ 워크로드 선택)
2.  **Git** 설치

## 1. 가상환경 생성 및 활성화 (PowerShell 기준)

```powershell
# 1. 가상환경 생성 (이름: d2env)
# Windows에서는 python3 대신 python을 주로 사용합니다.
# 시스템에 여러 버전이 있다면 py -3.10 처럼 버전을 명시하는 것이 좋습니다.
py -3.10 -m venv d2env

# 2. 가상환경 활성화
# Windows PowerShell 명령어입니다.
.\d2env\Scripts\Activate.ps1

# 성공 확인: 터미널 입력창 맨 앞에 (d2env)가 보이면 성공!
# 만약 보안 오류가 나면 다음 명령어를 먼저 실행하세요: Set-ExecutionPolicy -ExecutionPolicy RemoteSigned -Scope Process
```

## 2. 🛠️ 통합 라이브러리 설치

가상환경이 활성화된 상태에서 순서대로 실행하세요.

```powershell
# 0. pip 및 빌드 도구 업그레이드
python -m pip install --upgrade pip
# Ninja가 있으면: CPU 코어를 최대한 활용해서 설치 속도가 훨씬 빨라집니다.
pip install wheel ninja

# 1. PyTorch 2.7.1 설치 (CUDA 버전별 명령어)
# 사용자님의 CUDA 환경(12.1)에 맞는 명령어를 선택하세요.

# [추천] 사용자 CUDA 12.1 환경 (추정 명령어) 현재 노트북은 Cuda가 13.0이라 128로 변경해서 설치했음.
pip install torch==2.7.1 torchvision==0.22.1 torchaudio==2.7.1 --index-url https://download.pytorch.org/whl/cu121

# 2. SegFormer 및 필수 라이브러리
pip install transformers pillow numpy matplotlib opencv-python

# 3. Detectron2 설치 (Windows 호환 수정 버전)
# 주의: Detectron2는 Windows에서 바로 pip install로 설치하기 어렵습니다.
# 아래 과정을 따라주세요.

# 3-1. Detectron2 소스 다운로드 (이미 있다면 생략 가능) D:\git\detectron2\detectron2_repo
git clone https://github.com/facebookresearch/detectron2.git detectron2_repo

# 3-2. 컴파일 및 설치 (복잡한 과정이므로 아래 스크립트를 그대로 복사해서 실행하세요)
# Visual Studio 환경 변수를 불러와서 설치를 진행합니다.
$vs_script = Get-ChildItem -Path "C:\Program Files\Microsoft Visual Studio" -Recurse -Filter "vcvars64.bat" -ErrorAction SilentlyContinue | Select-Object -First 1 -ExpandProperty FullName

# Visual Studio 환경 변수를 불러와서 설치를 진행합니다.
cmd /c "call `"$vs_script`" && set DISTUTILS_USE_SDK=1 && set MSSdk=1 && pip install -e detectron2_repo --no-build-isolation"

# 4. YOLO 모델 사용을 위한 라이브러리
# Ultralytics는 PyTorch가 이미 설치되어 있으면 잘 설치됩니다.
pip install ultralytics

In [ ]:
import torch
print(f"PyTorch 버전: {torch.__version__}")
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")
print(f"CUDA 버전 (PyTorch 빌드): {torch.version.cuda}")
if torch.cuda.is_available():
    print(f"CUDA 디바이스: {torch.cuda.get_device_name(0)}")
else:
    print("CUDA 디바이스: N/A")

# Dectrono clone 
설치완료해서 모두 필요없음

In [ ]:
# !python -m pip install pyyaml==5.1
import sys, os, distutils.core
# Note: This is a faster way to install detectron2 in Colab, but it does not include all functionalities (e.g. compiled operators).
# See https://detectron2.readthedocs.io/tutorials/install.html for full installation instructions
!git clone 'https://github.com/facebookresearch/detectron2'  # Detectron2 GitHub 리포지토리를 현재 디렉토리에 복제합니다.
dist = distutils.core.run_setup("./detectron2/setup.py")  # 복제된 detectron2 디렉토리 내 setup.py를 실행하여 설치에 필요한 정보를 가져옵니다. (주로 install_requires 목록)
!python -m pip install {' '.join([f"'{x}'" for x in dist.install_requires])}  # setup.py에서 가져온 필수 의존성 패키지들을 pip을 사용하여 설치합니다.
sys.path.insert(0, os.path.abspath('./detectron2'))  # 복제된 detectron2 디렉토리를 Python 모듈 검색 경로(sys.path)의 맨 앞에 추가하여, 설치 없이 로컬에서 detectron2 모듈을 바로 가져와(import) 사용할 수 있도록 합니다.

# Properly install detectron2. (Please do not install twice in both ways)
# !python -m pip install 'git+https://github.com/facebookresearch/detectron2.git'

# 환경확인

In [ ]:
import torch
import sys

print("=" * 50)
print("환경 정보")
print("=" * 50)
print(f"Python 버전: {sys.version.split()[0]}")
print(f"PyTorch 버전: {torch.__version__}")
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA 버전: {torch.version.cuda}")
    print(f"GPU 디바이스: {torch.cuda.get_device_name(0)}")
    print(f"GPU 메모리: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("⚠️  CUDA를 사용할 수 없습니다. CPU 모드로 실행됩니다.")

print("=" * 50)

# Transformers 확인
try:
    import transformers
    print(f"✓ Transformers 버전: {transformers.__version__}")
except ImportError:
    print("❌ Transformers가 설치되지 않았습니다.")
    print("   설치: pip install transformers")

# . ADE20K 추론 및 시각화 코드 (test_ade20k.py)
아래 코드를 복사하여 실행해 보세요. IMAGE_PATH에 테스트하고 싶은 실내 사진(복도, 방 등) 경로만 넣어주면 됩니다.

# Oneformer_Semantic_Demo  아래 Panoptic 할것

In [ ]:
from transformers import OneFormerProcessor, OneFormerForUniversalSegmentation
from PIL import Image
import torch

# 로컬 이미지 사용 (이미 로드된 image 변수 사용)
# image = Image.open(IMAGE_PATH).convert("RGB")  # 이미 위에서 로드했다면 생략

# 모델 로드
processor = OneFormerProcessor.from_pretrained("shi-labs/oneformer_ade20k_swin_large")
model = OneFormerForUniversalSegmentation.from_pretrained("shi-labs/oneformer_ade20k_swin_large")

# GPU 사용 (있다면)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Semantic Segmentation (실내 장면에 가장 적합)
semantic_inputs = processor(images=image, task_inputs=["semantic"], return_tensors="pt")
semantic_inputs = {k: v.to(device) for k, v in semantic_inputs.items()}  # GPU로 이동

with torch.no_grad():
    semantic_outputs = model(**semantic_inputs)

# 후처리 (outputs -> semantic_outputs로 수정)
predicted_semantic_map = processor.post_process_semantic_segmentation(
    semantic_outputs, 
    target_sizes=[image.size[::-1]]
)[0]

pred_seg = predicted_semantic_map.cpu().numpy()
print(f"✓ 추론 완료: {pred_seg.shape}")

# Oneformer Panoptic

셀 1] Import & 이미지 경로

In [ ]:
import glob
import os
from PIL import Image

# ADE20K 이미지 검색
IMAGE_DIR = "D:/git/detectron2/ade20k_consistency/original_ade20k"
image_files = glob.glob(os.path.join(IMAGE_DIR, "*.jpg"))
IMAGE_PATH = image_files[0]  # 다른 이미지: [1], [2] 등으로 변경

print(f"📂 총 {len(image_files)}개 이미지 발견")
print(f"✓ 선택된 이미지: {IMAGE_PATH}")

[셀 2] 이미지 로드

In [ ]:
# 이미지 로드
if not os.path.exists(IMAGE_PATH):
    raise FileNotFoundError(f"이미지 없음: {IMAGE_PATH}")

image = Image.open(IMAGE_PATH).convert("RGB")
print(f"✓ 이미지 로드 완료: {image.size}")

[셀 3] 모델 로드 (Tiny)

In [ ]:
from transformers import OneFormerProcessor, OneFormerForUniversalSegmentation
import torch

print("🔧 모델 로드 중... (OneFormer Tiny)")

# Tiny 모델로 변경
model_name = "shi-labs/oneformer_ade20k_swin_tiny"
processor = OneFormerProcessor.from_pretrained(model_name)
model = OneFormerForUniversalSegmentation.from_pretrained(model_name)

# GPU 사용
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(f"✓ 모델 로드 완료 (디바이스: {device})")

[셀 4] Panoptic 추론

In [ ]:
from transformers import OneFormerProcessor, OneFormerForUniversalSegmentation
from PIL import Image
import torch

# model_name = "shi-labs/oneformer_ade20k_swin_large"  # 성능 우선 시

print(f"🔧 모델 로드 중: {model_name}")

# 모델 로드
processor = OneFormerProcessor.from_pretrained(model_name)
model = OneFormerForUniversalSegmentation.from_pretrained(model_name)

# GPU 사용
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"✓ 모델 로드 완료 (디바이스: {device})")

# Panoptic Segmentation
print("🚀 Panoptic 추론 실행 중...")
panoptic_inputs = processor(images=image, task_inputs=["panoptic"], return_tensors="pt")
panoptic_inputs = {k: v.to(device) for k, v in panoptic_inputs.items()}

with torch.no_grad():
    panoptic_outputs = model(**panoptic_inputs)

# 후처리
panoptic_result = processor.post_process_panoptic_segmentation(
    panoptic_outputs, 
    target_sizes=[image.size[::-1]]
)[0]

# 결과 추출
pred_seg = panoptic_result["segmentation"].cpu().numpy()
segments_info = panoptic_result["segments_info"]

print(f"✓ Panoptic 추론 완료: {pred_seg.shape}")
print(f"  검출된 세그먼트 수: {len(segments_info)}")

# 시각화 셀

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 시각화
plt.figure(figsize=(15, 5))

# 원본 이미지
plt.subplot(1, 2, 1)
plt.imshow(image)
plt.title("Original Image")
plt.axis('off')

# Panoptic 결과
plt.subplot(1, 2, 2)
plt.imshow(pred_seg, cmap='tab20')
plt.title(f"Panoptic Segmentation ({len(segments_info)} segments)")
plt.axis('off')

plt.tight_layout()
plt.show()

# 세그먼트 정보 출력
print("\n🎯 검출된 세그먼트 정보:")
for i, seg in enumerate(segments_info):
    print(f"  {i+1}. ID={seg['id']}, Label={seg['label_id']}, Area={seg.get('area', 'N/A')}")

# 최종 전체파일

# -------------------------------------------------
# 0️⃣ 기본 설정 (한글 폰트, ipywidgets)
# -------------------------------------------------
import matplotlib as mpl
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Javascript

# 한글 폰트 지정 → Glyph 경고 사라짐
mpl.rcParams["font.family"] = "Malgun Gothic"
mpl.rcParams["axes.unicode_minus"] = False

# -------------------------------------------------
# 1️⃣ 이미지 리스트 준비
# -------------------------------------------------
import glob, os
from PIL import Image

IMAGE_DIR = r"D:/git/detectron2/ade20k_consistency/original_ade20k"
image_files = sorted(glob.glob(os.path.join(IMAGE_DIR, "*.jpg")))
if not image_files:
    raise FileNotFoundError(f"'{IMAGE_DIR}'에 이미지가 없습니다.")

# 현재 인덱스 (전역 변수)
cur_idx = 0

def load_image(idx: int):
    """주어진 인덱스의 이미지를 PIL 객체와 파일 경로로 반환"""
    path = image_files[idx]
    img = Image.open(path).convert("RGB")
    return img, path

# -------------------------------------------------
# 2️⃣ OneFormer Tiny 모델 로드 (한 번만 실행)
# -------------------------------------------------
import torch
from transformers import OneFormerProcessor, OneFormerForUniversalSegmentation

model_name = "shi-labs/oneformer_ade20k_swin_tiny"   # Tiny 버전 (≈220 MB)
print(f"🔧 모델 로드 중: {model_name}")

processor = OneFormerProcessor.from_pretrained(model_name)
model = OneFormerForUniversalSegmentation.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"✓ 모델 로드 완료 (디바이스: {device})")

# -------------------------------------------------
# 3️⃣ 시각화 함수 (벽·바닥 강조 + 중앙 라벨)
# -------------------------------------------------
import numpy as np
from scipy import ndimage   # 중심점 계산에 사용

def visualize_result(original_image, seg_map):
    """원본 이미지와 벽·바닥 마스크를 비교하고, 각 영역 중앙에 라벨을 표시합니다."""
    # ADE20K 클래스 ID: 0 = Wall, 3 = Floor
    colors = {0: [255, 0, 0, 120],   # 빨강 (벽)
              3: [0, 255, 0, 120]}   # 초록 (바닥)
    class_names = {0: "Wall (벽)", 3: "Floor (바닥)"}

    # RGBA 마스크 생성
    mask_overlay = np.zeros((seg_map.shape[0], seg_map.shape[1], 4), dtype=np.uint8)
    centroids = {}
    detected = []

    for cid, col in colors.items():
        matches = (seg_map == cid)
        if np.any(matches):
            mask_overlay[matches] = col
            detected.append(class_names[cid])

            # 중심점 계산
            y, x = np.where(matches)
            centroids[cid] = (int(x.mean()), int(y.mean()))

    # 원본을 RGBA로 변환 후 마스크 합성
    orig_rgba = np.array(original_image.convert("RGBA"))
    combined = Image.fromarray(orig_rgba)
    mask_img = Image.fromarray(mask_overlay)
    combined.alpha_composite(mask_img)

    # 시각화
    plt.figure(figsize=(15, 8))

    # 원본
    plt.subplot(1, 2, 1)
    plt.title("Original Image", fontsize=14)
    plt.imshow(original_image)
    plt.axis("off")

    # 결과 + 라벨
    plt.subplot(1, 2, 2)
    plt.title(f"ADE20K Result: {', '.join(detected)}", fontsize=14)
    plt.imshow(combined)

    # 라벨을 중앙에 표시
    for cid, (cx, cy) in centroids.items():
        plt.text(cx, cy, class_names[cid].split()[0],
                 color="white", fontsize=20, fontweight="bold",
                 ha="center", va="center",
                 bbox=dict(facecolor="black", alpha=0.7, edgecolor="white", linewidth=2))

    # 간단 범례 (텍스트)
    plt.text(10, 10, "Red: Wall (벽)\nGreen: Floor (바닥)",
             color="white", fontsize=12, fontweight="bold",
             bbox=dict(facecolor="black", alpha=0.5))

    plt.axis("off")
    plt.tight_layout()
    plt.show()
    print("✅ 시각화 완료!")

# -------------------------------------------------
# 4️⃣ 추론 + 시각화 함수 (이미지 인덱스에 따라 자동 실행)
# -------------------------------------------------
def run_inference_on_index(idx: int):
    """주어진 인덱스의 이미지에 대해 Panoptic 추론 후 시각화"""
    global cur_idx
    cur_idx = idx

    img, path = load_image(idx)
    print(f"\n📂 [{idx+1}/{len(image_files)}] 이미지: {os.path.basename(path)}")

    # ------------------- 추론 -------------------
    inputs = processor(images=img, task_inputs=["panoptic"], return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    panoptic_result = processor.post_process_panoptic_segmentation(
        outputs, target_sizes=[img.size[::-1]]
    )[0]

    seg_map = panoptic_result["segmentation"].cpu().numpy()
    segments_info = panoptic_result["segments_info"]
    print(f"✓ 추론 완료 – 검출된 세그먼트 수: {len(segments_info)}")

    # ------------------- 시각화 -------------------
    visualize_result(img, seg_map)

# -------------------------------------------------
# 5️⃣ 인터랙티브 UI (A/D 키 + 버튼)
# -------------------------------------------------
def _js_key_listener():
    """브라우저에 JavaScript를 삽입해 A/D 키를 감지하고 Python 콜백을 호출합니다."""
    js = """
    document.addEventListener('keydown', function(event) {
        // A (←) → 이전, D (→) → 다음
        if (event.key === 'a' || event.key === 'ArrowLeft') {
            IPython.notebook.kernel.execute('navigate_prev()');
        } else if (event.key === 'd' || event.key === 'ArrowRight') {
            IPython.notebook.kernel.execute('navigate_next()');
        }
    });
    """
    display(Javascript(js))

def navigate_prev():
    """이전 이미지로 이동 (버튼/키)"""
    new_idx = (cur_idx - 1) % len(image_files)
    run_inference_on_index(new_idx)

def navigate_next():
    """다음 이미지로 이동 (버튼/키)"""
    new_idx = (cur_idx + 1) % len(image_files)
    run_inference_on_index(new_idx)

# 초기 실행 (첫 번째 이미지)
run_inference_on_index(0)

# UI: 버튼 + 키 리스너
prev_btn = widgets.Button(description="← Previous (A)", layout=widgets.Layout(width='180px'))
next_btn = widgets.Button(description="Next (D) →", layout=widgets.Layout(width='180px'))

prev_btn.on_click(lambda _: navigate_prev())
next_btn.on_click(lambda _: navigate_next())

display(widgets.HBox([prev_btn, next_btn]))
_js_key_listener()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from scipy import ndimage
# Windows 기본 한글 폰트 (Malgun Gothic) 사용 → 한글 경고 사라짐
import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rcParams["font.family"] = "Malgun Gothic"
mpl.rcParams["axes.unicode_minus"] = False   # 마이너스 기호가 깨지는 경우 방지
def visualize_result(original_image, seg_map):
    """원본과 벽/바닥만 강조된 이미지를 비교하여 보여줍니다."""
    
    # 마스크용 빈 이미지 생성 (RGBA)
    mask_overlay = np.zeros((seg_map.shape[0], seg_map.shape[1], 4), dtype=np.uint8)
    
    # 색상 정의 (R, G, B, Alpha)
    colors = {
        0: [255, 0, 0, 120],    # 벽 (Wall): 빨간색 반투명
        3: [0, 255, 0, 120]     # 바닥 (Floor): 초록색 반투명
    }
    
    class_names = {
        0: "Wall",
        3: "Floor"
    }
    
    detected_classes = []
    centroids = {}  # 중심점 저장

    # 벽과 바닥만 칠하기 + 중심점 계산
    for class_id, color in colors.items():
        matches = (seg_map == class_id)
        if np.any(matches):
            mask_overlay[matches] = color
            label_name = "Wall (벽)" if class_id == 0 else "Floor (바닥)"
            detected_classes.append(label_name)
            
            # 중심점(Centroid) 계산
            y_coords, x_coords = np.where(matches)
            centroid_y = int(np.mean(y_coords))
            centroid_x = int(np.mean(x_coords))
            centroids[class_id] = (centroid_x, centroid_y)

    # 원본 이미지를 numpy로 변환
    original_np = np.array(original_image.convert("RGBA"))
    
    # 원본 위에 마스크 합성
    combined_img = Image.fromarray(original_np)
    mask_img = Image.fromarray(mask_overlay)
    combined_img.alpha_composite(mask_img)

    # 그래프로 출력
    plt.figure(figsize=(15, 10))
    
    plt.subplot(1, 2, 1)
    plt.title("Original Image", fontsize=14)
    plt.imshow(original_image)
    plt.axis('off')
    
    plt.subplot(1, 2, 2)
    plt.title(f"ADE20K Result: {', '.join(detected_classes)}", fontsize=14)
    plt.imshow(combined_img)
    
    # 클래스 이름을 중앙에 표시
    for class_id, (cx, cy) in centroids.items():
        label = class_names[class_id]
        color = 'white'
        plt.text(cx, cy, label, 
                color=color, fontsize=20, fontweight='bold',
                ha='center', va='center',
                bbox=dict(facecolor='black', alpha=0.7, edgecolor='white', linewidth=2))
    
    # 범례 추가
    plt.text(10, 10, "Red: Wall (벽)\nGreen: Floor (바닥)", 
             color='white', fontsize=12, fontweight='bold', 
             bbox=dict(facecolor='black', alpha=0.5))
    
    plt.axis('off')
    plt.tight_layout()
    plt.show()
    print("✅ 완료! 창을 확인하세요.")
# 시각화 함수 호출
visualize_result(image, pred_seg)

# 세그먼트 상세 정보 출력
print("\n🎯 검출된 세그먼트 상세 정보:")
for i, seg in enumerate(segments_info):
    print(f"  {i+1}. Segment ID={seg['id']}, Label ID={seg['label_id']}, Area={seg.get('area', 'N/A')}")

In [ ]:
run_inference()

In [ ]:
# Some basic setup:
# Setup detectron2 logger
import detectron2
from detectron2.utils.logger import setup_logger
setup_logger()

# import some common libraries
import numpy as np
import os, json, cv2, random

# 이미지 저장 경로 설정
picture_dir = "/home/elicer/dev/dj/picture"
os.makedirs(picture_dir, exist_ok=True)

# import some common detectron2 utilities
from detectron2 import model_zoo
from detectron2.engine import DefaultPredictor
from detectron2.config import get_cfg
from detectron2.utils.visualizer import Visualizer
from detectron2.data import MetadataCatalog, DatasetCatalog

# Panoptic Code

In [ ]:
# Some basic setup:
# Setup detectron2 logger
import detectron2
from detectron2.utils.logger import setup_logger
setup_logger()

# import some common libraries
import numpy as np
import os, json, cv2, random

# 이미지 저장 경로 설정
picture_dir = "/home/elicer/dev/dj/picture"
os.makedirs(picture_dir, exist_ok=True)

# import some common detectron2 utilities
from detectron2 import model_zoo
from detectron2.engine import DefaultPredictor
from detectron2.config import get_cfg
from detectron2.utils.visualizer import Visualizer
from detectron2.data import MetadataCatalog, DatasetCatalog

# Run a pre-trained detectron2 model

We first download an image from the COCO dataset:

In [ ]:
!wget http://images.cocodataset.org/val2017/000000439715.jpg -q -O input.jpg
im = cv2.imread("./input.jpg")

cv2.imwrite(os.path.join(picture_dir, "01_input_image.jpg"), im)
print(f"이미지 저장: {os.path.join(picture_dir, '01_input_image.jpg')}")
# cv2_imshow(im)

Then, we create a detectron2 config and a detectron2 `DefaultPredictor` to run inference on this image.

In [ ]:
# ========================================
# ADE20K 시맨틱 세그멘테이션 자동 테스트 코드
# ========================================

import os
import cv2
import glob
import numpy as np
from detectron2.config import get_cfg
from detectron2.engine import DefaultPredictor
from detectron2.utils.visualizer import Visualizer
from detectron2.data import MetadataCatalog

# ========================================
# 1. 로컬 ADE20K 이미지 파일 찾기
# ========================================
# ADE20K Consistency set에서 이미지 파일 검색
IMAGE_DIR = "D:/git/detectron2/ade20k_consistency/original_ade20k"
image_files = glob.glob(os.path.join(IMAGE_DIR, "*.jpg"))

# 첫 번째 이미지 선택 (또는 원하는 번호로 변경 가능)
IMAGE_PATH = image_files[0]  # 다른 이미지를 보고 싶으면 [0]을 [1], [2] 등으로 변경
print(f"📂 총 {len(image_files)}개 이미지 발견")
print(f"✓ 선택된 이미지: {IMAGE_PATH}")

# ========================================
# 2. 이미지 로드 (OpenCV 사용)
# ========================================
# OpenCV는 BGR 형식으로 이미지를 로드합니다
img = cv2.imread(IMAGE_PATH)
print(f"✓ 이미지 로드 완료: {img.shape}")  # (높이, 너비, 채널)



In [ ]:
import os
from PIL import Image

print(f"📂 이미지 처리 중: {IMAGE_PATH}")

# 파일 존재 확인
if not os.path.exists(IMAGE_PATH):
    print(f"❌ 오류: 이미지 파일을 찾을 수 없습니다.")
    print(f"   경로: {IMAGE_PATH}")
else:
    # PIL Image로 로드 (SegFormer는 PIL 이미지 사용)
    try:
        image = Image.open(IMAGE_PATH).convert("RGB")
        print(f"✓ 이미지 로드 완료: {image.size}")
    except Exception as e:
        print(f"❌ 오류: 이미지를 읽을 수 없습니다 - {e}")

In [ ]:
try:
    processor = SegformerImageProcessor.from_pretrained("nvidia/segformer-b5-finetuned-ade-512-512")
    model = SegformerForSemanticSegmentation.from_pretrained("nvidia/segformer-b5-finetuned-ade-512-512")
except Exception as e:
    print(f"❌ 오류: 모델 로드 중 오류 발생: {str(e)}")

In [ ]:
# NVIDIA/SegFormer 모델 로드 (b0는 경량, b5는 고성능. 여기선 b5 사용)
from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation
from PIL import Image
import torch
import numpy as np
import matplotlib.pyplot as plt

# ================= 설정 =================
# 테스트할 이미지 경로 (여기를 수정하세요!)
IMAGE_PATH = "test_room.jpg" 

# ADE20K에서 벽과 바닥의 클래스 ID (HuggingFace SegFormer 기준)
# 0: wall (벽), 3: floor (바닥)
TARGET_IDS = [0, 3] 
# =======================================
def run_inference():
    """ADE20K SegFormer 모델을 사용한 세그멘테이션 추론"""
    
    print("1. 모델을 로드하는 중입니다... (ADE20K Pre-trained)")

    
    processor = SegformerImageProcessor.from_pretrained("nvidia/segformer-b5-finetuned-ade-512-512")
    model = SegformerForSemanticSegmentation.from_pretrained("nvidia/segformer-b5-finetuned-ade-512-512")

    print(f"2. 이미지 처리 중: {IMAGE_PATH}")
    
    # 파일 존재 확인
    if not os.path.exists(IMAGE_PATH):
        print(f"❌ 오류: 이미지 파일을 찾을 수 없습니다.")
        print(f"   경로: {IMAGE_PATH}")
        return
    
    # PIL Image로 로드 (SegFormer는 PIL 이미지 사용)
    try:
        image = Image.open(IMAGE_PATH).convert("RGB")
        print(f"✓ 이미지 로드 완료: {image.size}")
    except Exception as e:
        print(f"❌ 오류: 이미지를 읽을 수 없습니다 - {e}")
        return

    # 추론 (Inference)
    print("   추론 실행 중...")
    inputs = processor(images=image, return_tensors="pt")
    outputs = model(**inputs)
    logits = outputs.logits  # shape (batch_size, num_labels, height/4, width/4)

    # 결과를 이미지 크기로 업샘플링
    upsampled_logits = torch.nn.functional.interpolate(
        logits,
        size=image.size[::-1],  # (height, width)
        mode="bilinear",
        align_corners=False,
    )

    # 가장 높은 확률의 클래스 선택 (Segmentation Map 생성)
    pred_seg = upsampled_logits.argmax(dim=1)[0]
    pred_seg = pred_seg.detach().cpu().numpy()

    # === 시각화 ===
    print("3. 결과 시각화 생성 중...")
    visualize_result(image, pred_seg)

def visualize_result(original_image, seg_map):
    """원본과 벽/바닥만 강조된 이미지를 비교하여 보여줍니다."""
    
    # 마스크용 빈 이미지 생성 (RGBA)
    mask_overlay = np.zeros((seg_map.shape[0], seg_map.shape[1], 4), dtype=np.uint8)
    
    # 색상 정의 (R, G, B, Alpha)
    colors = {
        0: [255, 0, 0, 120],    # 벽 (Wall): 빨간색 반투명
        3: [0, 255, 0, 120]     # 바닥 (Floor): 초록색 반투명
    }
    
    detected_classes = []

    # 벽과 바닥만 칠하기
    for class_id, color in colors.items():
        # 해당 클래스인 픽셀 찾기
        matches = (seg_map == class_id)
        if np.any(matches):
            mask_overlay[matches] = color
            label_name = "Wall (벽)" if class_id == 0 else "Floor (바닥)"
            detected_classes.append(label_name)

    # 원본 이미지를 numpy로 변환
    original_np = np.array(original_image.convert("RGBA"))
    
    # 원본 위에 마스크 합성
    combined_img = Image.fromarray(original_np)
    mask_img = Image.fromarray(mask_overlay)
    combined_img.alpha_composite(mask_img)

    # 그래프로 출력
    plt.figure(figsize=(15, 10))
    
    plt.subplot(1, 2, 1)
    plt.title("Original Image")
    plt.imshow(original_image)
    plt.axis('off')
    
    plt.subplot(1, 2, 2)
    plt.title(f"ADE20K Result: {', '.join(detected_classes)}")
    plt.imshow(combined_img)
    
    # 범례 추가 (HTML 색상 코드 사용 불가하므로 텍스트로 대체)
    plt.text(10, 10, "Red: Wall (벽)\nGreen: Floor (바닥)", 
             color='white', fontsize=12, fontweight='bold', 
             bbox=dict(facecolor='black', alpha=0.5))
    
    plt.axis('off')
    plt.show()
    print("✅ 완료! 창을 확인하세요.")

if __name__ == "__main__":
    run_inference()

In [ ]:
if __name__ == "__main__":
    run_inference()

In [ ]:
from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation
from PIL import Image
import torch
import numpy as np
import matplotlib.pyplot as plt

# ================= 설정 =================
# 테스트할 이미지 경로 (여기를 수정하세요!)
IMAGE_PATH = "test_room.jpg" 

# ADE20K에서 벽과 바닥의 클래스 ID (HuggingFace SegFormer 기준)
# 0: wall (벽), 3: floor (바닥)
TARGET_IDS = [0, 3] 
# =======================================

def run_inference():
    print("1. 모델을 로드하는 중입니다... (ADE20K Pre-trained)")
    # NVIDIA/SegFormer 모델 로드 (b0는 경량, b5는 고성능. 여기선 b5 사용)
    processor = SegformerImageProcessor.from_pretrained("nvidia/segformer-b5-finetuned-ade-512-512")
    model = SegformerForSemanticSegmentation.from_pretrained("nvidia/segformer-b5-finetuned-ade-512-512")

    print(f"2. 이미지 처리 중: {IMAGE_PATH}")
    try:
        image = Image.open(IMAGE_PATH)
    except FileNotFoundError:
        print("❌ 오류: 이미지 파일을 찾을 수 없습니다. 경로를 확인해주세요.")
        return

    # 추론 (Inference)
    inputs = processor(images=image, return_tensors="pt")
    outputs = model(**inputs)
    logits = outputs.logits  # shape (batch_size, num_labels, height/4, width/4)

    # 결과를 이미지 크기로 업샘플링
    upsampled_logits = torch.nn.functional.interpolate(
        logits,
        size=image.size[::-1], # (height, width)
        mode="bilinear",
        align_corners=False,
    )

    # 가장 높은 확률의 클래스 선택 (Segmentation Map 생성)
    pred_seg = upsampled_logits.argmax(dim=1)[0]
    pred_seg = pred_seg.detach().cpu().numpy()

    # === 시각화 ===
    print("3. 결과 시각화 생성 중...")
    visualize_result(image, pred_seg)

def visualize_result(original_image, seg_map):
    """원본과 벽/바닥만 강조된 이미지를 비교하여 보여줍니다."""
    
    # 마스크용 빈 이미지 생성 (RGBA)
    mask_overlay = np.zeros((seg_map.shape[0], seg_map.shape[1], 4), dtype=np.uint8)
    
    # 색상 정의 (R, G, B, Alpha)
    colors = {
        0: [255, 0, 0, 120],    # 벽 (Wall): 빨간색 반투명
        3: [0, 255, 0, 120]     # 바닥 (Floor): 초록색 반투명
    }
    
    detected_classes = []

    # 벽과 바닥만 칠하기
    for class_id, color in colors.items():
        # 해당 클래스인 픽셀 찾기
        matches = (seg_map == class_id)
        if np.any(matches):
            mask_overlay[matches] = color
            label_name = "Wall (벽)" if class_id == 0 else "Floor (바닥)"
            detected_classes.append(label_name)

    # 원본 이미지를 numpy로 변환
    original_np = np.array(original_image.convert("RGBA"))
    
    # 원본 위에 마스크 합성
    combined_img = Image.fromarray(original_np)
    mask_img = Image.fromarray(mask_overlay)
    combined_img.alpha_composite(mask_img)

    # 그래프로 출력
    plt.figure(figsize=(15, 10))
    
    plt.subplot(1, 2, 1)
    plt.title("Original Image")
    plt.imshow(original_image)
    plt.axis('off')
    
    plt.subplot(1, 2, 2)
    plt.title(f"ADE20K Result: {', '.join(detected_classes)}")
    plt.imshow(combined_img)
    
    # 범례 추가 (HTML 색상 코드 사용 불가하므로 텍스트로 대체)
    plt.text(10, 10, "Red: Wall (벽)\nGreen: Floor (바닥)", 
             color='white', fontsize=12, fontweight='bold', 
             bbox=dict(facecolor='black', alpha=0.5))
    
    plt.axis('off')
    plt.show()
    print("✅ 완료! 창을 확인하세요.")

if __name__ == "__main__":
    run_inference()

In [ ]:

# ========================================
# 4. 추론 (Inference) 실행
# ========================================
print("🚀 추론 실행 중...")
outputs = predictor(img)  # 이미지를 입력하여 세그멘테이션 결과 얻기
print("✓ 추론 완료")

# outputs["instances"]에는 다음 정보가 포함됩니다:
# - pred_boxes: 검출된 객체의 바운딩 박스
# - scores: 각 객체의 신뢰도 점수
# - pred_classes: 각 객체의 클래스 (예: 침대, 의자, 창문 등)
# - pred_masks: 각 객체의 픽셀 단위 마스크

# ========================================
# 5. 결과 시각화
# ========================================
# Visualizer: 결과를 시각적으로 표현하는 도구
# - img[:, :, ::-1]: BGR → RGB 변환 (OpenCV는 BGR, Matplotlib는 RGB 사용)
# - MetadataCatalog: 클래스 이름, 색상 등의 메타데이터 제공
# - scale: 텍스트 및 마스크 크기 조정 (1.2 = 120%)
v = Visualizer(img[:, :, ::-1], 
               MetadataCatalog.get(cfg.DATASETS.TRAIN[0]), 
               scale=1.2)

# 검출된 객체(instances)를 이미지 위에 그리기
out = v.draw_instance_predictions(outputs["instances"].to("cpu"))

# ========================================
# 6. 결과 표시 및 저장
# ========================================
import matplotlib.pyplot as plt

# 그래프 크기 설정 (12인치 x 8인치)
plt.figure(figsize=(12, 8))

# 시각화된 이미지 표시
plt.imshow(out.get_image())
plt.axis('off')  # 축 눈금 제거
plt.title("ADE20K Semantic Segmentation Result", fontsize=16)

# 결과 이미지 저장 (고해상도)
OUTPUT_PATH = "./result_ade20k.png"
plt.savefig(OUTPUT_PATH, bbox_inches='tight', dpi=150)
print(f"✓ 결과 저장: {OUTPUT_PATH}")

# 화면에 표시
plt.show()

# ========================================
# 7. 검출된 객체 정보 출력 (선택사항)
# ========================================
print("\n🎯 검출된 객체 정보:")
instances = outputs["instances"].to("cpu")
classes = instances.pred_classes.numpy()
scores = instances.scores.numpy()

# 메타데이터에서 클래스 이름 가져오기
metadata = MetadataCatalog.get(cfg.DATASETS.TRAIN[0])
class_names = metadata.thing_classes

for i, (cls, score) in enumerate(zip(classes, scores)):
    print(f"  {i+1}. {class_names[cls]}: {score*100:.1f}%")

In [ ]:
cfg = get_cfg()
# add project-specific config (e.g., TensorMask) here if you're not running a model in detectron2's core library
cfg.merge_from_file(model_zoo.get_config_file("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"))
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5  # set threshold for this model
# Find a model from detectron2's model zoo. You can use the https://dl.fbaipublicfiles... url as well
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml")
predictor = DefaultPredictor(cfg)
outputs = predictor(im)

In [ ]:
# look at the outputs. See https://detectron2.readthedocs.io/tutorials/models.html#model-output-format for specification
print(outputs["instances"].pred_classes)
print(outputs["instances"].pred_boxes)

In [ ]:
# We can use `Visualizer` to draw the predictions on the image.
v = Visualizer(im[:, :, ::-1], MetadataCatalog.get(cfg.DATASETS.TRAIN[0]), scale=1.2)
out = v.draw_instance_predictions(outputs["instances"].to("cpu"))
output_img = out.get_image()[:, :, ::-1]
cv2.imwrite(os.path.join(picture_dir, "02_instance_segmentation_result.jpg"), output_img)
print(f"이미지 저장: {os.path.join(picture_dir, '02_instance_segmentation_result.jpg')}")

# Train on a custom dataset

In this section, we show how to train an existing detectron2 model on a custom dataset in a new format.

We use [the balloon segmentation dataset](https://github.com/matterport/Mask_RCNN/tree/master/samples/balloon)
which only has one class: balloon.
We'll train a balloon segmentation model from an existing model pre-trained on COCO dataset, available in detectron2's model zoo.

Note that COCO dataset does not have the "balloon" category. We'll be able to recognize this new class in a few minutes.

## Prepare the dataset

In [ ]:
# download, decompress the data
!wget https://github.com/matterport/Mask_RCNN/releases/download/v2.1/balloon_dataset.zip
!unzip balloon_dataset.zip > /dev/null

Register the balloon dataset to detectron2, following the [detectron2 custom dataset tutorial](https://detectron2.readthedocs.io/tutorials/datasets.html).
Here, the dataset is in its custom format, therefore we write a function to parse it and prepare it into detectron2's standard format. User should write such a function when using a dataset in custom format. See the tutorial for more details.


In [ ]:
# if your dataset is in COCO format, this cell can be replaced by the following three lines:
# from detectron2.data.datasets import register_coco_instances
# register_coco_instances("my_dataset_train", {}, "json_annotation_train.json", "path/to/image/dir")
# register_coco_instances("my_dataset_val", {}, "json_annotation_val.json", "path/to/image/dir")

from detectron2.structures import BoxMode

def get_balloon_dicts(img_dir):
    json_file = os.path.join(img_dir, "via_region_data.json")
    with open(json_file) as f:
        imgs_anns = json.load(f)

    dataset_dicts = []
    for idx, v in enumerate(imgs_anns.values()):
        record = {}

        filename = os.path.join(img_dir, v["filename"])
        height, width = cv2.imread(filename).shape[:2]

        record["file_name"] = filename
        record["image_id"] = idx
        record["height"] = height
        record["width"] = width

        annos = v["regions"]
        objs = []
        for _, anno in annos.items():
            assert not anno["region_attributes"]
            anno = anno["shape_attributes"]
            px = anno["all_points_x"]
            py = anno["all_points_y"]
            poly = [(x + 0.5, y + 0.5) for x, y in zip(px, py)]
            poly = [p for x in poly for p in x]

            obj = {
                "bbox": [np.min(px), np.min(py), np.max(px), np.max(py)],
                "bbox_mode": BoxMode.XYXY_ABS,
                "segmentation": [poly],
                "category_id": 0,
            }
            objs.append(obj)
        record["annotations"] = objs
        dataset_dicts.append(record)
    return dataset_dicts

for d in ["train", "val"]:
    DatasetCatalog.register("balloon_" + d, lambda d=d: get_balloon_dicts("balloon/" + d))
    MetadataCatalog.get("balloon_" + d).set(thing_classes=["balloon"])
balloon_metadata = MetadataCatalog.get("balloon_train")

To verify the dataset is in correct format, let's visualize the annotations of randomly selected samples in the training set:



In [ ]:
dataset_dicts = get_balloon_dicts("balloon/train")
for idx, d in enumerate(random.sample(dataset_dicts, 3)):
    img = cv2.imread(d["file_name"])
    visualizer = Visualizer(img[:, :, ::-1], metadata=balloon_metadata, scale=0.5)
    out = visualizer.draw_dataset_dict(d)
    output_img = out.get_image()[:, :, ::-1]
    save_path = os.path.join(picture_dir, f"03_dataset_visualization_{idx+1}.jpg")
    cv2.imwrite(save_path, output_img)
    print(f"이미지 저장: {save_path}")

## Train!

Now, let's fine-tune a COCO-pretrained R50-FPN Mask R-CNN model on the balloon dataset. It takes ~2 minutes to train 300 iterations on a P100 GPU.


In [ ]:
from detectron2.engine import DefaultTrainer

cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"))
cfg.DATASETS.TRAIN = ("balloon_train",)
cfg.DATASETS.TEST = ()
cfg.DATALOADER.NUM_WORKERS = 2
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml")  # Let training initialize from model zoo
cfg.SOLVER.IMS_PER_BATCH = 2  # This is the real "batch size" commonly known to deep learning people
cfg.SOLVER.BASE_LR = 0.00025  # pick a good LR
cfg.SOLVER.MAX_ITER = 300    # 300 iterations seems good enough for this toy dataset; you will need to train longer for a practical dataset
cfg.SOLVER.STEPS = []        # do not decay learning rate
cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 128   # The "RoIHead batch size". 128 is faster, and good enough for this toy dataset (default: 512)
cfg.MODEL.ROI_HEADS.NUM_CLASSES = 1  # only has one class (ballon). (see https://detectron2.readthedocs.io/tutorials/datasets.html#update-the-config-for-new-datasets)
# NOTE: this config means the number of classes, but a few popular unofficial tutorials incorrect uses num_classes+1 here.

os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)
trainer = DefaultTrainer(cfg)
trainer.resume_or_load(resume=False)
trainer.train()

In [ ]:
# Look at training curves in tensorboard:
%load_ext tensorboard
%tensorboard --logdir output

## Inference & evaluation using the trained model
Now, let's run inference with the trained model on the balloon validation dataset. First, let's create a predictor using the model we just trained:



In [ ]:
# Inference should use the config with parameters that are used in training
# cfg now already contains everything we've set previously. We changed it a little bit for inference:
cfg.MODEL.WEIGHTS = os.path.join(cfg.OUTPUT_DIR, "model_final.pth")  # path to the model we just trained
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.7   # set a custom testing threshold
predictor = DefaultPredictor(cfg)

Then, we randomly select several samples to visualize the prediction results.

In [ ]:
from detectron2.utils.visualizer import ColorMode
dataset_dicts = get_balloon_dicts("balloon/val")
for idx, d in enumerate(random.sample(dataset_dicts, 3)):
    im = cv2.imread(d["file_name"])
    outputs = predictor(im)  # format is documented at https://detectron2.readthedocs.io/tutorials/models.html#model-output-format
    v = Visualizer(im[:, :, ::-1],
                   metadata=balloon_metadata,
                   scale=0.5,
                   instance_mode=ColorMode.IMAGE_BW   # remove the colors of unsegmented pixels. This option is only available for segmentation models
    )
    out = v.draw_instance_predictions(outputs["instances"].to("cpu"))
    output_img = out.get_image()[:, :, ::-1]
    save_path = os.path.join(picture_dir, f"04_validation_result_{idx+1}.jpg")
    cv2.imwrite(save_path, output_img)
    print(f"이미지 저장: {save_path}")

We can also evaluate its performance using AP metric implemented in COCO API.
This gives an AP of ~70. Not bad!

In [ ]:
from detectron2.evaluation import COCOEvaluator, inference_on_dataset
from detectron2.data import build_detection_test_loader
evaluator = COCOEvaluator("balloon_val", output_dir="./output")
val_loader = build_detection_test_loader(cfg, "balloon_val")
print(inference_on_dataset(predictor.model, val_loader, evaluator))
# another equivalent way to evaluate the model is to use `trainer.test`

# Other types of builtin models

We showcase simple demos of other types of models below:

In [ ]:
# Inference with a keypoint detection model
cfg = get_cfg()   # get a fresh new config
cfg.merge_from_file(model_zoo.get_config_file("COCO-Keypoints/keypoint_rcnn_R_50_FPN_3x.yaml"))
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.7  # set threshold for this model
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url("COCO-Keypoints/keypoint_rcnn_R_50_FPN_3x.yaml")
predictor = DefaultPredictor(cfg)
outputs = predictor(im)
v = Visualizer(im[:,:,::-1], MetadataCatalog.get(cfg.DATASETS.TRAIN[0]), scale=1.2)
out = v.draw_instance_predictions(outputs["instances"].to("cpu"))
output_img = out.get_image()[:, :, ::-1]
cv2.imwrite(os.path.join(picture_dir, "05_keypoint_detection_result.jpg"), output_img)
print(f"이미지 저장: {os.path.join(picture_dir, '05_keypoint_detection_result.jpg')}")

In [ ]:
# Inference with a panoptic segmentation model
cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file("COCO-PanopticSegmentation/panoptic_fpn_R_101_3x.yaml"))
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url("COCO-PanopticSegmentation/panoptic_fpn_R_101_3x.yaml")
predictor = DefaultPredictor(cfg)
panoptic_seg, segments_info = predictor(im)["panoptic_seg"]
v = Visualizer(im[:, :, ::-1], MetadataCatalog.get(cfg.DATASETS.TRAIN[0]), scale=1.2)
out = v.draw_panoptic_seg_predictions(panoptic_seg.to("cpu"), segments_info)
output_img = out.get_image()[:, :, ::-1]
cv2.imwrite(os.path.join(picture_dir, "06_panoptic_segmentation_result.jpg"), output_img)
print(f"이미지 저장: {os.path.join(picture_dir, '06_panoptic_segmentation_result.jpg')}")

# Run panoptic segmentation on a video

In [ ]:
# This is the video we're going to process
from IPython.display import YouTubeVideo, display
video = YouTubeVideo("ll8TgCZ0plk", width=500)
display(video)

In [ ]:
# Install dependencies, download the video, and crop 5 seconds for processing
!pip install youtube-dl
!youtube-dl https://www.youtube.com/watch?v=ll8TgCZ0plk -f 22 -o video.mp4
!ffmpeg -i video.mp4 -t 00:00:06 -c:v copy video-clip.mp4

In [ ]:
# Run frame-by-frame inference demo on this video (takes 3-4 minutes) with the "demo.py" tool we provided in the repo.
!git clone https://github.com/facebookresearch/detectron2
# Note: this is currently BROKEN due to missing codec. See https://github.com/facebookresearch/detectron2/issues/2901 for workaround.
%run detectron2/demo/demo.py --config-file detectron2/configs/COCO-PanopticSegmentation/panoptic_fpn_R_101_3x.yaml --video-input video-clip.mp4 --confidence-threshold 0.6 --output video-output.mkv \
  --opts MODEL.WEIGHTS detectron2://COCO-PanopticSegmentation/panoptic_fpn_R_101_3x/139514519/model_final_cafdb1.pkl

In [ ]:
# 결과는 video-output.mkv 파일로 저장됩니다
# 리눅스 환경에서는 파일을 직접 확인하거나 scp 등으로 다운로드하세요
print(f"비디오 결과 저장: video-output.mkv")